# EDA 001.06 — Other Data Cleaning Techniques

**Beyond the Kaggle Data Cleaning Course**

This notebook covers practical data cleaning techniques not included in the [Kaggle Data Cleaning course](https://www.kaggle.com/learn/data-cleaning):

1. **Outlier Detection & Handling**
2. **Duplicate Detection & Removal**
3. **Data Type Coercion**
4. **Reshaping Data** (wide ↔ long)
5. **Cross-field Validation**
6. **Mixed-type Columns**
7. **Text Data Cleaning**
8. **High Cardinality Categorical Columns**

---

### Popular Datasets to Practise Data Cleaning

| Dataset | Source | Why it's useful |
|---|---|---|
| [Titanic](https://www.kaggle.com/c/titanic) | Kaggle | Missing values, mixed types, encoding |
| [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) | Kaggle | 79 features, many with nulls, skewed distributions |
| [NYC Taxi Trips](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) | NYC TLC | Outliers, datetime parsing, duplicates |
| [Adult Income (Census)](https://archive.ics.uci.edu/ml/datasets/adult) | UCI ML Repo | Inconsistent categories, mixed types |
| [Brazilian E-Commerce (Olist)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) | Kaggle | Multi-table, text, dates, high cardinality |
| [OpenFoodFacts](https://world.openfoodfacts.org/data) | Open Food Facts | Large, real-world messiness across all cleaning topics |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## 1 — Outlier Detection & Handling

**Reference:** [Towards Data Science — Ways to Detect and Remove Outliers](https://towardsdatascience.com/ways-to-detect-and-remove-the-outliers-404d16608dba)

Outliers are data points that differ significantly from the rest. Common strategies:

| Method | When to use |
|---|---|
| **IQR Rule** | Skewed distributions, robust to extreme values |
| **Z-score** | Roughly normal distributions |
| **Capping (Winsorisation)** | Preserve row count, limit extreme influence |
| **Drop** | Only when outlier is clearly erroneous |

> **Note:** Never blindly drop outliers — they may carry genuine signal (e.g. fraud detection).

In [ ]:
# --- Sample data ---
np.random.seed(42)
df = pd.DataFrame({'salary': np.append(np.random.normal(50000, 8000, 200), [5, 200000, 999999])})

# --- IQR Method ---
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers_iqr = df[(df['salary'] < lower) | (df['salary'] > upper)]
print(f"IQR bounds: [{lower:.0f}, {upper:.0f}]")
print(f"Outliers detected (IQR): {len(outliers_iqr)}")

# --- Z-score Method ---
z_scores = (df['salary'] - df['salary'].mean()) / df['salary'].std()
outliers_z = df[z_scores.abs() > 3]
print(f"Outliers detected (Z-score > 3): {len(outliers_z)}")

# --- Capping ---
df['salary_capped'] = df['salary'].clip(lower=lower, upper=upper)

# --- Visualise ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['salary'].plot(kind='box', ax=axes[0], title='Before Capping')
df['salary_capped'].plot(kind='box', ax=axes[1], title='After Capping (IQR)')
plt.tight_layout()
plt.show()

---
## 2 — Duplicate Detection & Removal

**Reference:** [Pandas docs — `DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

| Scenario | Approach |
|---|---|
| Exact duplicates | `df.duplicated()` + `df.drop_duplicates()` |
| Subset duplicates | `df.duplicated(subset=['col1', 'col2'])` |
| Keep first / last / none | `keep='first'`, `'last'`, `False` |

In [ ]:
df = pd.DataFrame({
    'id':   [1, 2, 2, 3, 4, 4],
    'name': ['Alice', 'Bob', 'Bob', 'Carol', 'Dave', 'Dave'],
    'score': [90, 85, 85, 78, 92, 99],  # row 4/5 differ on score — subset dup on id+name only
})

print(f"Total rows: {len(df)}")
print(f"Exact duplicates: {df.duplicated().sum()}")
print(f"Subset duplicates (id+name): {df.duplicated(subset=['id', 'name']).sum()}")

# Drop exact duplicates
df_clean = df.drop_duplicates()
print(f"\nAfter drop_duplicates(): {len(df_clean)} rows")

# Drop subset duplicates, keep last
df_clean2 = df.drop_duplicates(subset=['id', 'name'], keep='last')
print(f"After subset drop (keep last): {len(df_clean2)} rows")
df_clean2

---
## 3 — Data Type Coercion

**Reference:** [Pandas docs — `DataFrame.astype`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

Pandas often infers types incorrectly when loading CSV data. Common cases:
- Numeric columns loaded as `object` (string) due to stray characters (`$`, `,`, `%`)
- Boolean columns stored as `int` (0/1)
- Category columns left as `object` (wastes memory)

> **Tip:** Use `pd.to_numeric(..., errors='coerce')` to safely convert — invalid values become `NaN` instead of raising an error.

In [ ]:
df = pd.DataFrame({
    'price':    ['$1,200', '$850', '$3,000', 'N/A'],
    'is_active': [1, 0, 1, 0],
    'category':  ['A', 'B', 'A', 'C'],
})
print("Before:")
print(df.dtypes, "\n")

# Strip currency symbols/commas, coerce
df['price'] = pd.to_numeric(df['price'].str.replace(r'[$,]', '', regex=True), errors='coerce')

# Boolean
df['is_active'] = df['is_active'].astype(bool)

# Category (memory efficient for low-cardinality columns)
df['category'] = df['category'].astype('category')

print("After:")
print(df.dtypes)
df

---
## 4 — Reshaping Data (Wide ↔ Long)

**Reference:** [Pandas docs — Reshaping](https://pandas.pydata.org/docs/user_guide/reshaping.html)

| Operation | Method | Use case |
|---|---|---|
| Wide → Long | `df.melt()` | Tidy data for analysis/plotting |
| Long → Wide | `df.pivot()` / `df.pivot_table()` | Reporting, feature matrices |
| Multi-index | `df.stack()` / `df.unstack()` | Hierarchical index manipulation |

In [ ]:
# Wide format — one row per subject, columns per year
wide = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Carol'],
    '2022':    [85, 90, 78],
    '2023':    [88, 92, 80],
    '2024':    [91, 89, 84],
})
print("Wide format:")
display(wide)

# Wide → Long (tidy)
long = wide.melt(id_vars='student', var_name='year', value_name='score')
long['year'] = long['year'].astype(int)
print("\nLong (tidy) format:")
display(long.head(6))

# Long → Wide (pivot back)
wide_back = long.pivot(index='student', columns='year', values='score').reset_index()
print("\nPivoted back to wide:")
display(wide_back)

---
## 5 — Cross-field Validation

Consistency checks between **related columns** — constraints that should hold true by domain logic.

Examples:
- `start_date` must be before `end_date`
- `age` must be positive and plausible (e.g. < 120)
- `discount_price` ≤ `original_price`
- `num_children` should be 0 if `marital_status == 'single'` (soft check)

In [ ]:
df = pd.DataFrame({
    'id':         [1, 2, 3, 4],
    'start_date': pd.to_datetime(['2024-01-01', '2024-05-10', '2024-03-01', '2024-07-20']),
    'end_date':   pd.to_datetime(['2024-06-01', '2024-04-01', '2024-08-15', '2024-09-01']),
    'age':        [25, -3, 130, 42],
    'price':      [100, 80, 60, 50],
    'discount':   [90, 90, 50, 70],
})

# Violations
bad_dates   = df[df['start_date'] >= df['end_date']]
bad_age     = df[(df['age'] < 0) | (df['age'] > 120)]
bad_price   = df[df['discount'] > df['price']]

print(f"Rows where start_date >= end_date : {len(bad_dates)}")
print(f"Rows with implausible age         : {len(bad_age)}")
print(f"Rows where discount > price       : {len(bad_price)}")

# Combined violation flag
df['has_violation'] = (
    (df['start_date'] >= df['end_date']) |
    (df['age'] < 0) | (df['age'] > 120) |
    (df['discount'] > df['price'])
)
df

---
## 6 — Mixed-type Columns

**Reference:** [Pandas docs — Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)

Happens when a column intended to be numeric contains stray strings (`'N/A'`, `'unknown'`, `'-'`, empty strings). Pandas loads the whole column as `object`.

> **Pattern:** Extract numeric part with regex → `pd.to_numeric(..., errors='coerce')` → decide on NaN strategy.

In [ ]:
df = pd.DataFrame({
    'revenue': ['1200', '850.5', 'N/A', '', 'unknown', '3000', '-', '500'],
})

print("dtype before:", df['revenue'].dtype)

# Coerce to numeric — non-numeric become NaN
df['revenue_clean'] = pd.to_numeric(df['revenue'], errors='coerce')

print(f"dtype after : {df['revenue_clean'].dtype}")
print(f"NaNs introduced: {df['revenue_clean'].isna().sum()}")
df

---
## 7 — Text Data Cleaning

**References:**
- [Pandas — String Methods](https://pandas.pydata.org/docs/user_guide/text.html)
- [Python `re` module docs](https://docs.python.org/3/library/re.html)

Common tasks:

| Task | Method |
|---|---|
| Strip whitespace | `.str.strip()` |
| Normalize case | `.str.lower()` / `.str.title()` |
| Remove special characters | `.str.replace(r'[^\w\s]', '', regex=True)` |
| Strip HTML tags | regex or `BeautifulSoup` |
| Normalize unicode | `unicodedata.normalize('NFKD', ...)` |

In [ ]:
df = pd.DataFrame({
    'review': [
        '  Great product!!! 😊  ',
        '<b>Terrible</b> — would NOT buy again...',
        'AMAZING quality, fast shipping!!!',
        '  ok i guess ???  ',
    ]
})

df['clean'] = (
    df['review']
    .str.strip()                                        # remove leading/trailing whitespace
    .str.replace(r'<[^>]+>', '', regex=True)            # strip HTML tags
    .str.replace(r'[^\w\s]', ' ', regex=True)           # remove punctuation / special chars
    .str.replace(r'\s+', ' ', regex=True)               # collapse multiple spaces
    .str.strip()
    .str.lower()
)

df

---
## 8 — High Cardinality Categorical Columns

**Reference:** [Feature Engineering for Machine Learning — Alice Zheng & Amanda Casari (O'Reilly)](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/)

High cardinality = many unique values (e.g. city, job title, product name). Direct one-hot encoding explodes dimensionality.

| Strategy | When to use |
|---|---|
| **Group rare categories** | Keep top-N, map rest to `'Other'` |
| **Frequency encoding** | Replace category with its count / proportion |
| **Target encoding** | Replace with mean of target — risk of leakage, use with CV |
| **Hashing** | Fixed-size representation, fast, some collision risk |

In [ ]:
np.random.seed(0)
cities = ['New York', 'Mumbai', 'London', 'Delhi', 'Tokyo',
          'Paris', 'Sydney', 'Toronto', 'Berlin', 'Lagos']
# Simulate skewed distribution — few cities dominate
weights = [0.30, 0.20, 0.15, 0.10, 0.08, 0.06, 0.04, 0.03, 0.02, 0.02]
df = pd.DataFrame({'city': np.random.choice(cities, size=500, p=weights)})

# --- Strategy 1: Group rare categories (keep top N) ---
top_n = 4
top_cities = df['city'].value_counts().nlargest(top_n).index
df['city_grouped'] = df['city'].where(df['city'].isin(top_cities), other='Other')
print("Grouped value counts:")
print(df['city_grouped'].value_counts())

# --- Strategy 2: Frequency encoding ---
freq_map = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq_map)

print(f"\nFrequency encoding sample:")
df[['city', 'city_grouped', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False)